In [3]:
%%file model.py

from accelerate import Accelerator
#accelerator = Accelerator(mixed_precision="fp16")
accelerator = Accelerator()

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision.transforms import v2 as T
from tqdm import tqdm
import time
from torchvision.io import read_image, ImageReadMode
import evaluate
from accelerate.utils import set_seed
from torchvision.ops import sigmoid_focal_loss
import random

seed = 42
set_seed(seed)
random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


def seed_worker(worker_id):
    worker_seed = seed
    random.seed(worker_seed)








def detect_environment():
    if "COLAB_GPU" in os.environ or "COLAB_TPU_ADDR" in os.environ:
        return "Colab"
    elif "KAGGLE_URL_BASE" in os.environ:
        return "Kaggle"
    else:
        return "Local"


def create_metrics():
    metrics = {
        "accuracy": evaluate.load("accuracy"),
        "precision": evaluate.load("precision"),
        "recall": evaluate.load("recall"),
        "f1": evaluate.load("f1"),
    }
    return metrics


def update_metrics(metrics, preds, labels):
    for metric in metrics.values():
        metric.add_batch(predictions=preds, references=labels)



def compute_metrics(metrics):
    results = {k: v.compute()[k] for k, v in metrics.items()}
    return results











class DDDSequenceDataset(Dataset):

    def __init__(self, root_dir, time_window_in_seconds, total_frames_in_window,
                 time_window_right_shift_by_seconds, transform):

        self.root_dir = root_dir
        self.time_window_in_seconds = time_window_in_seconds
        self.total_frames_in_window = total_frames_in_window
        self.time_window_right_shift_by_seconds = time_window_right_shift_by_seconds
        self.transform = transform

        self.total_positive = 0
        self.total_negative = 0
        self.sequence_frame_paths_and_labels = []
       
        self._sequence_processing()
        self.total_windows = len(self.sequence_frame_paths_and_labels)

        print(f"Total sequences: {self.total_windows}")
        print(f"Positive sequences: {self.total_positive}")
        print(f"Negative sequences: {self.total_negative}")


    def _sequence_processing(self):

        for subject in sorted(os.listdir(self.root_dir)):
            subject_path = os.path.join(self.root_dir, subject)

            if not os.path.isdir(subject_path):
                continue

            for scenario in sorted(os.listdir(subject_path)):
                video_path = os.path.join(subject_path, scenario)

                if not os.path.isdir(video_path):
                    continue

                frame_files = sorted(
                    f for f in os.listdir(video_path)
                    if f.lower().endswith(".jpg")
                )

                total_frames_in_video = len(frame_files)
                if total_frames_in_video == 0:
                    continue

                video_fps = 15 if scenario in ["night_noglasses", "night_glasses"] else 30

                window_frame_count = min(
                    total_frames_in_video,
                    int(video_fps * self.time_window_in_seconds)
                )

                video_stride = max(
                    1,
                    int(round(self.time_window_right_shift_by_seconds * video_fps))
                )

                total_windows_in_video = max(
                    1,
                    (total_frames_in_video - window_frame_count) // video_stride + 1
                )

                frame_time_interval = (
                    self.time_window_in_seconds /
                    self.total_frames_in_window
                )

                for window_index in range(total_windows_in_video):

                    frame_paths = []
                    frame_labels = []

                    for i in range(self.total_frames_in_window):

                        t = (
                            window_index * self.time_window_right_shift_by_seconds +
                            i * frame_time_interval
                        )

                        frame_index = min(
                            int(round(t * video_fps)),
                            total_frames_in_video - 1
                        )

                        fname = frame_files[frame_index]
                        frame_path = os.path.join(video_path, fname)

                        frame_paths.append(frame_path)

                        label = int(fname.rsplit("_", 1)[-1].split(".")[0])
                        frame_labels.append(label)

                    sequence_label = frame_labels[-1]

                    if sequence_label == 1:
                        self.total_positive += 1
                    else:
                        self.total_negative += 1

                    self.sequence_frame_paths_and_labels.append((frame_paths, sequence_label))

    def get_pos_weight(self):
        pos_weight = self.total_negative / max(self.total_positive, 1)
        return torch.tensor([pos_weight], dtype=torch.float32)


    def __len__(self):
        return self.total_windows


    def __getitem__(self, idx):

        frame_paths, label = self.sequence_frame_paths_and_labels[idx]
        images = []

        for path in frame_paths:
            img = read_image(path, mode=ImageReadMode.GRAY).float() / 255.0
            img = img.repeat(3, 1, 1)
            
            if self.transform:
                img = self.transform(img)

            images.append(img)

        images = torch.stack(images)  # (T, C, H, W)
        return images, torch.tensor(label, dtype=torch.float32)





import torch
import torch.nn as nn
import timm


class ViTTemporalModel(nn.Module):

    def __init__(self, vit_model="vit_tiny_patch16_224",  temporal_layers=2, heads=3, dropout=0.3, max_frames=32):
        super().__init__()
        self.vit = timm.create_model(
            vit_model,
            pretrained=True,
            num_classes=0  
        )

        embed_dim = self.vit.num_features

        self.temporal_pos = nn.Parameter(
            torch.zeros(1, max_frames, embed_dim)
        )
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            batch_first=True,
            activation="gelu"
        )

        self.temporal_transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=temporal_layers
        )

        self.temporal_attention = nn.Sequential(
            nn.Linear(embed_dim, 1),
            nn.Softmax(dim=1)
        )

        self.classifier = nn.Linear(embed_dim, 1)

    def forward(self, x):
        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W)
        features = self.vit(x)
        features = features.view(B, T, -1)
        features = features + self.temporal_pos[:, :T]
        temporal_features = self.temporal_transformer(features)
        weights = self.temporal_attention(temporal_features)
        pooled = (temporal_features * weights).sum(dim=1)
        logits = self.classifier(pooled)
        return logits.squeeze(1)




    
model = ViTTemporalModel(
    vit_model="vit_tiny_patch16_224",
    temporal_layers=2,
    heads=3,
    dropout=0.3,
    max_frames=32
)

for param in model.vit.parameters():
    param.requires_grad = False

for param in model.vit.blocks[-1].parameters():
    param.requires_grad = True


batch_size = 16
frames = 24
embed_dim = 64
patch_size = 8
lr = 3e-5
weight_decay = 0.05

batch_size=32
time_window_in_seconds=8
total_frames_in_window=24
time_window_right_shift_by_seconds=5

transform = T.Compose([
    T.RandomHorizontalFlip(),
    T.RandomRotation(degrees=10),
    T.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
])

# /kaggle/input/datasets/manith04/ddd-processed-1-training-frames-type-1
# /kaggle/input/datasets/manith04/ddd-processed-2-training-frames-type-1
# /kaggle/input/datasets/manith04/ddd-processed-validation-frames-type-1


print("[HERE 0]")


train_dataset_1 = DDDSequenceDataset(root_dir="/kaggle/input/datasets/manith04/ddd-processed-1-training-frames-type-1", 
                                     time_window_in_seconds=time_window_in_seconds,
                                     total_frames_in_window=total_frames_in_window,
                                     time_window_right_shift_by_seconds=time_window_right_shift_by_seconds,
                                     transform=transform) 

train_dataset_2 = DDDSequenceDataset(root_dir="/kaggle/input/datasets/manith04/ddd-processed-2-training-frames-type-1",
                                     time_window_in_seconds=time_window_in_seconds,
                                     total_frames_in_window=total_frames_in_window,
                                     time_window_right_shift_by_seconds=time_window_right_shift_by_seconds,
                                     transform=transform) 

validation_dataset = DDDSequenceDataset(root_dir="/kaggle/input/datasets/manith04/ddd-processed-validation-frames-type-1", 
                                        time_window_in_seconds=time_window_in_seconds,
                                        total_frames_in_window=total_frames_in_window,
                                        time_window_right_shift_by_seconds=time_window_right_shift_by_seconds,
                                        transform=None) 


print("[HERE 1]")

cpu_count = os.cpu_count()

train_dataset = ConcatDataset([train_dataset_1, train_dataset_2])
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=cpu_count, pin_memory=True, 
                          persistent_workers=True, prefetch_factor=(6), in_order=True, worker_init_fn=seed_worker
) 
                   
val_loader = DataLoader(validation_dataset, batch_size=batch_size , shuffle=False, num_workers=cpu_count, pin_memory=True, 
                        persistent_workers=True, prefetch_factor=(6), in_order=True,  worker_init_fn=seed_worker)


print("[HERE 2]")



optimizer = torch.optim.AdamW([
    {"params": model.vit.blocks[-1].parameters(), "lr": 1e-5},
    {"params": model.temporal_transformer.parameters(), "lr": 3e-5},
    {"params": model.temporal_attention.parameters(), "lr": 3e-5},
    {"params": model.classifier.parameters(), "lr": 3e-5},
], weight_decay=0.05)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',           
    factor=0.5,           # multiply LR by 0.5 when triggered
    patience=5,           
    min_lr=1e-7          
)

print("[HERE 3]")

model, optimizer, train_loader, val_loader = accelerator.prepare(model, optimizer, train_loader, val_loader)
device = accelerator.device


total_positive = train_dataset_1.total_positive + train_dataset_2.total_positive
total_negative = train_dataset_1.total_negative + train_dataset_2.total_negative
pos_weight = torch.tensor(min(total_negative / max(total_positive, 1), 10.0),  # cap at 10
                            dtype=torch.float32,  device=device)

# criterion = sigmoid_focal_loss(alpha=alpha, gamma=2.0)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)


epochs = 120
checkpoint_dir = "checkpoint"
best_model_path = "best_model_new.pth"
log_file = "metrics_log.txt"
extra_path = os.path.join(checkpoint_dir, "extra_state.pt")

best_val_loss = float("inf")
resume_step, start_epoch = 0, 0


print("[HERE 4]")


if os.path.isdir(checkpoint_dir) and len(os.listdir(checkpoint_dir)) > 0:
    accelerator.print("🔄 Loading checkpoint...")
    accelerator.load_state(checkpoint_dir)

    if os.path.exists(extra_path):

        extra = torch.load(extra_path, map_location=device)
        start_epoch = extra.get("epoch", 0)
        best_val_loss = extra.get("best_val_loss", float("inf"))
        accelerator.print(f"Resuming from epoch {start_epoch}")

    else:
        accelerator.print("⚡ No extra_state.pt found, starting from scratch.")


if accelerator.is_main_process and start_epoch == 0:
    with open(log_file, "w") as file:
        file.write("epoch,train_loss,train_acc,train_precision,train_recall,train_f1,val_loss,val_acc,val_precision,val_recall,val_f1\n")

print("[HERE 5]")


for epoch in range(start_epoch, epochs):
    model.train()
    
    if accelerator.is_main_process:
        print(f"\nEpoch {epoch+1}/{epochs}")

    train_metrics = create_metrics()
    val_metrics = create_metrics()

    train_loss = 0
    train_bar =  tqdm(train_loader, desc="Training", disable=not accelerator.is_main_process)
    
    for frames, labels in train_bar:
        frames, labels = frames.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(frames)
        if torch.isnan(outputs).any():
            print("NaN right after forward pass")
            print(outputs.min(), outputs.max())

        
        outputs = outputs.clamp(-20, 20)

        #loss = sigmoid_focal_loss(inputs=outputs.squeeze(), targets=labels.float(), alpha=alpha, gamma=2.0, reduction='mean')
        loss = criterion(outputs, labels)
        accelerator.backward(loss)

        if torch.isnan(loss):
            print("LOSS NAN")
        
        if torch.isnan(outputs).any():
            print("OUTPUT NAN")
        

        if accelerator.sync_gradients:
            accelerator.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss += loss.item()
        preds = (torch.sigmoid(outputs) > 0.5).int()
        preds, labels = accelerator.gather_for_metrics((preds, labels.int()))

        update_metrics(train_metrics, preds, labels)

    train_results = compute_metrics(train_metrics)
    avg_train_loss = train_loss / len(train_loader)

    if accelerator.is_main_process:
        print(
            f"Epoch {epoch+1} - "
            f"Train Loss: {avg_train_loss:.4f} | "
            f"Acc: {train_results['accuracy']:.4f} | "
            f"Precision: {train_results['precision']:.4f} | "
            f"Recall: {train_results['recall']:.4f} | "
            f"F1: {train_results['f1']:.4f}"
        )

    model.eval()
    val_loss = 0
    
    with torch.no_grad():
        val_bar = tqdm(val_loader, desc="Validation", disable=not accelerator.is_main_process)
        
        for frames, labels in val_bar:    
            frames, labels = frames.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            outputs = model(frames)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            preds = (torch.sigmoid(outputs) > 0.5).int()
            preds, labels = accelerator.gather_for_metrics((preds, labels.int()))

            update_metrics(val_metrics, preds, labels)

    val_results = compute_metrics(val_metrics)
    avg_val_loss = val_loss / len(val_loader)
    scheduler.step(val_results["accuracy"])  # scheduler updated based on val accuracy

    if accelerator.is_main_process:
        print(
            f"Epoch {epoch+1} - "
            f"Val Loss: {avg_val_loss:.4f} | "
            f"Acc: {val_results['accuracy']:.4f} | "
            f"Precision: {val_results['precision']:.4f} | "
            f"Recall: {val_results['recall']:.4f} | "
            f"F1: {val_results['f1']:.4f}"
        )

        with open(log_file, "a") as f:
            f.write(
                f"{epoch+1},{avg_train_loss:.3f},{train_results['accuracy']:.3f},"
                f"{train_results['precision']:.3f},{train_results['recall']:.3f},{train_results['f1']:.3f},"
                f"{avg_val_loss:.3f},{val_results['accuracy']:.3f},{val_results['precision']:.3f},"
                f"{val_results['recall']:.3f},{val_results['f1']:.3f}\n"
            )

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            unwrapped_model = accelerator.unwrap_model(model)
            torch.save(unwrapped_model.state_dict(), best_model_path)
            print("⭐ Best model updated!")

        accelerator.save_state(checkpoint_dir)
        torch.save({"epoch": epoch + 1, "best_val_loss": best_val_loss}, extra_path)

Overwriting model.py


In [14]:
# !rm -rf /kaggle/working

rm: cannot remove '/kaggle/working': Device or resource busy


In [ ]:
!uv pip install evaluate -q
!accelerate launch model.py --num_processes=2

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `2`
		More than one GPU was found, enabling multi-GPU training.
		If this was unintended please pass in `--num_processes=1`.
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(
[HERE 0]
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(
[HERE 0]
Total sequences: 2879
Positive seq